# Dataset filtering

In [ ]:
import pandas as pd
import geopandas as gpd
from andeangc import config as cfg

## Data

In [ ]:
# read data from previous processing step
AndeanGC_data     = pd.read_csv(cfg.VERSION / 'AndeanGC_data.csv').set_index('date')
AndeanGC_metadata = pd.read_csv(cfg.VERSION / 'AndeanGC_metadata.csv').set_index('gauge_id')
AndeanGC_shape    = gpd.read_file(cfg.VERSION / 'AndeanGC_shape.gpkg').set_index('gauge_id')
AndeanGC_shape["basin_area"] = AndeanGC_shape.to_crs(32719).area / 1e6  # UTM 19S, the Andes

## Basins selection

In [ ]:
# basins with glacier area > 0.1% (RGI v6.0)
rgi_version = cfg.rgi_version
rgi = pd.concat([gpd.read_file(cfg.data_dir('glacier_outlines') / f"{rgi_version}_16.shp"), 
                 gpd.read_file(cfg.data_dir('glacier_outlines') / f"{rgi_version}_17.shp")])
rgi = rgi.rename(columns={"RGIId": "rgi_id"}).set_index("rgi_id")
rgi_union = rgi.union_all()

AndeanGC_shape = AndeanGC_shape[AndeanGC_shape.intersects(rgi_union)].copy()
AndeanGC_shape['glacier_area_RGI60'] = AndeanGC_shape.intersection(rgi_union).to_crs(epsg=32719).area / 1e6
AndeanGC_shape = AndeanGC_shape.fillna(0)  # fill NaN values with 0

# > 0.1% glacier area
AndeanGC_shape['glacier_area_RGI60'] = (AndeanGC_shape.glacier_area_RGI60 * 100 / AndeanGC_shape.basin_area)
AndeanGC_shape = AndeanGC_shape[AndeanGC_shape.glacier_area_RGI60 > cfg.glacier_threshold]

AndeanGC_data = AndeanGC_data[AndeanGC_shape.index]
AndeanGC_metadata = AndeanGC_metadata.loc[AndeanGC_shape.index]
AndeanGC_metadata = pd.concat([AndeanGC_metadata, AndeanGC_shape[["basin_area", "glacier_area_RGI60"]]], axis=1)

In [ ]:
# basins with more than 1 year of data (365 days)
AndeanGC_data = AndeanGC_data.interpolate(method='linear', limit=cfg.interpolation_limit) # first interpolate (stations without data over weekends)
AndeanGC_metadata["days_w_data"] = AndeanGC_data.notna().sum()
AndeanGC_metadata = AndeanGC_metadata[AndeanGC_metadata.days_w_data > cfg.min_data_days]

In [ ]:
# basins with "minimal" intervention (quality check is next step to remove more)
keywords_remove = "|".join(cfg.intervention_keywords)  # single source: config.yml
AndeanGC_metadata = AndeanGC_metadata[~AndeanGC_metadata["gauge_name"].str.contains(keywords_remove, case=False, na=False)]
AndeanGC_data = AndeanGC_data[AndeanGC_metadata.index]
AndeanGC_shape = AndeanGC_shape.loc[AndeanGC_metadata.index]

## Save

In [ ]:
AndeanGC_metadata.to_csv(cfg.VERSION / 'AndeanGC_metadata.csv')
AndeanGC_data.to_csv(cfg.VERSION / 'AndeanGC_data.csv')
AndeanGC_shape.to_file(cfg.VERSION / 'AndeanGC_shape.gpkg')